# Install dependencies not already in colab and imports

In [ ]:
!pip -q install keras-tuner

import os
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    top_k_accuracy_score
)

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.layers import Dropout
from keras.optimizers import SGD

import keras_tuner

DATA_DIR = Path("/content/data")
FIGURES_DIR = Path("/content/figures")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
np.random.seed(0)
tf.random.set_seed(0)


def find_or_upload_file(filename):
    """Return a path to filename. In Colab, prompts upload if the file is missing."""
    candidates = [
        DATA_DIR / filename,
        Path("/content") / filename,
        Path(filename),
    ]

    for path in candidates:
        if path.exists():
            target = DATA_DIR / filename
            if path.resolve() != target.resolve():
                shutil.copy(path, target)
            return target

    try:
        from google.colab import files
        print(f"Upload {filename} when prompted.")
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(
                f"Expected {filename}, but uploaded: {list(uploaded.keys())}"
            )
        target = DATA_DIR / filename
        shutil.move(filename, target)
        return target
    except ModuleNotFoundError:
        raise FileNotFoundError(
            f"Could not find {filename}. Put it in {DATA_DIR} or the current directory."
        )


def data_path(filename):
    return DATA_DIR / filename


def figure_path(filename):
    return FIGURES_DIR / filename


# Preprocessing Step and Colab File Setup

Upload `original_dataset.csv` when prompted. The notebook saves all generated CSVs, the label encoder, and figures into `/content/data` and `/content/figures`.


In [ ]:
df = pd.read_csv(find_or_upload_file('original_dataset.csv'))


# check for missing values 
print(df.isnull().sum())
# in our case there are none 

print(df.info())
print(df.describe())

# all genre value counts are reasonable in range to one another 
# but since not excat use stratify=y to preserve genre proportions in train/test split
print(df['Genre'].value_counts())

# Genre
# pop       1161
# rock      1136
# latin     1036
# hiphop     971
# edm        956
# rap        926
# r&b        731

# drop unnecessary columns 
# ex. drop time_signature since almost all songs in 4, some in 3 but won't help us predict genre
# same with other features, they will just add noise 
df = df.drop(columns=['Title', 'Album_cover_link', 'Artist', 'duration_ms', 'explicit', 'id', 
                      'release_date', 'release_date_precision', 'total_tracks', 'mode', 'time_signature'])


# now we just have the features we want 
print(df.columns)

feature_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[feature_cols].hist(figsize=(12, 12), bins=15)

plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.savefig(figure_path('feature_distributions.png'), dpi=300, bbox_inches="tight")

plt.show()

plt.figure(figsize=(10, 8))

correlation_mat = df[feature_cols].corr()

sns.heatmap(correlation_mat, annot=True, cmap='coolwarm')

plt.title("Feature Correlation Matrix")
plt.savefig(figure_path('feature_correlation_matrix.png'), dpi=300, bbox_inches="tight")
plt.show()

X = df.drop(columns=['Genre'])
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Genre'])

# we save our label encoder so the model files can use it 
# to interpret class labels later
joblib.dump(label_encoder, data_path('label_encoder.pkl'))

# we use stratify so that genre proportions are preserved in train/test split
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# convert scaled arrays back to data frames 
X_train_scaled_df = pd.DataFrame(
    X_train_scaled, 
    columns=X.columns
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X.columns
)

# convert labels to data frames 
Y_train_df = pd.DataFrame(Y_train, columns=['Genre'])
Y_test_df = pd.DataFrame(Y_test, columns=['Genre'])

X_train_scaled_df.to_csv(data_path('X_train_scaled.csv'), index=False)

X_test_scaled_df.to_csv(data_path('X_test_scaled.csv'), index=False)

Y_train_df.to_csv(data_path('Y_train.csv'), index=False)

Y_test_df.to_csv(data_path('Y_test.csv'), index=False)


print(f"Saved processed data to: {DATA_DIR}")
print(f"Saved figures to: {FIGURES_DIR}")


Judging from this correlation matrix, it doesn't appear that we have severe multicollinearity. Therefore, our features are mostly contributing unique information and it makes sense to include them all in our models. 

The strongest positive correlation is energy and loudness at r = 0.61 while the strongest negative correlation appears to be energy and acousticness at r = -0.49. This makes sense because energetic sounds tend to be louder, while acousticness songs tend to be quieter and slower-paced. 

Other features with decent correlation are danceability and valence, since happier songs tend to be more danceable, and danceability with speechiness. 

Due to there not being feature redundancy, we will elect to not perform PCA since there is no need for dimension reduction in this project. 

We elect to use the StandardScaler method from sklearn instead of MinMaxScaler because SVM's work better with centered data, and kNN distance calculations are more balanced this way. 

We also use the standard 80/20 split for train/test. We scale after the split as well, and then export the datasets for the models to use

# Logistic Regression Model 

In [ ]:
X_train = pd.read_csv(data_path('X_train_scaled.csv'))
X_test = pd.read_csv(data_path('X_test_scaled.csv'))
Y_train = pd.read_csv(data_path('Y_train.csv')) 
Y_test = pd.read_csv(data_path('Y_test.csv'))
genre_names = [
    "EDM",
    "HipHop",
    "Latin",
    "Pop",
    "R&B",
    "Rap",
    "Rock"
]
y_train = Y_train.values.ravel()
y_test = Y_test.values.ravel()

# Logistic Regression model using L2 regularization
model1 = LogisticRegression(
    penalty='l2',
    C=0.3,
    solver='saga',
    max_iter=1000,
    random_state=0
)

model1.fit(X_train, y_train)
y_pred = model1.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# per-class precision/recall/F1 + overall summary
print(
    classification_report(
        y_test,
        y_pred,
        target_names=genre_names
    )
)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=genre_names,
    yticklabels=genre_names
)

plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.title("Logistic Regression using L2's Confusion Matrix")

plt.tight_layout()
plt.show()

# Logistic Regression using 'L1' regularization
model = LogisticRegression(
    penalty='l1',
    C=0.3,
    solver='saga',
    max_iter=1000,
    random_state=0
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy)
print(
    classification_report(
        y_test,
        y_pred,
        target_names=genre_names
    )
)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=genre_names,
    yticklabels=genre_names
)

plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.title("Logistic Regression using L1's Confusion Matrix")

plt.tight_layout()
plt.show()

# An array from 10^-2 to 10^2
lambda_values = np.logspace(-2, 2, 100)

train_accuracies = []
test_accuracies = []

for l in lambda_values:

    model = LogisticRegression(
        penalty='l1',
        C=1/l,
        solver='saga',
        max_iter=1000,
        random_state=0
    )

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_accuracies.append(
        accuracy_score(y_train, y_train_pred)
    )

    test_accuracies.append(
        accuracy_score(y_test, y_test_pred)
    )

print("Best Train Accuracy:", max(train_accuracies))
print("Best Test Accuracy:", max(test_accuracies))

plt.figure(figsize=(8, 5))

plt.plot(
    lambda_values,
    train_accuracies,
    label='Train Accuracy'
)

plt.plot(
    lambda_values,
    test_accuracies,
    label='Test Accuracy'
)

plt.grid(True)

plt.xscale('log')

plt.xlabel("λ")
plt.ylabel("Accuracy")

plt.title("Train vs Test Accuracy against different λ values(L1)")

plt.legend()

plt.show()

lambda_values = np.logspace(-2, 2, 100)

train_accuracies = []
test_accuracies = []

for l in lambda_values:

    model = LogisticRegression(
        penalty='l2',
        C=1/l,
        solver='saga',
        max_iter=1000,
        random_state=0
    )

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_accuracies.append(
        accuracy_score(y_train, y_train_pred)
    )

    test_accuracies.append(
        accuracy_score(y_test, y_test_pred)
    )

print("Best Train Accuracy:", max(train_accuracies))
print("Best Test Accuracy:", max(test_accuracies))

plt.figure(figsize=(8, 5))

plt.plot(
    lambda_values,
    train_accuracies,
    label='Train Accuracy'
)

plt.plot(
    lambda_values,
    test_accuracies,
    label='Test Accuracy'
)

plt.grid(True)

plt.xscale('log')

plt.xlabel("λ")
plt.ylabel("Accuracy")

plt.title("Train vs Test Accuracy against different λ values(L2)")

plt.legend()

plt.show()

from itertools import combinations
from sklearn.metrics import accuracy_score, top_k_accuracy_score

c_values = np.logspace(-2, 2, 10)
results = []

columns = list(X_train.columns)
# Check the combination of {1/λ, number of columns dropped} can give a better model performance
for c in c_values:
    # 2-4 columns are dropped
    for k in range(2, 5):
        # Iterate each combination of features given k dropped columns
        for cols_to_drop in combinations(columns, k):

            X_train_drop = X_train.drop(columns=list(cols_to_drop))
            X_test_drop = X_test.drop(columns=list(cols_to_drop))

            model = LogisticRegression(
                C=c,
                solver='saga',
                max_iter=1000,
                random_state=0
            )

            model.fit(X_train_drop, y_train)

            y_train_pred = model.predict(X_train_drop)
            y_pred = model.predict(X_test_drop)

            y_train_prob = model.predict_proba(X_train_drop)
            y_prob = model.predict_proba(X_test_drop)

            acc_train = accuracy_score(y_train, y_train_pred)
            acc = accuracy_score(y_test, y_pred)

            top3_train = top_k_accuracy_score(
                y_train,
                y_train_prob,
                k=3
            )

            top3_test = top_k_accuracy_score(
                y_test,
                y_prob,
                k=3
            )

            # Append result in 
            results.append({
                'C': c,
                'k': k,
                'Dropped Features': cols_to_drop,
                'Train Accuracy': acc_train,
                'Test Accuracy': acc,
                'Train Top3 Accuracy': top3_train,
                'Test Top3 Accuracy': top3_test
            })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='Test Accuracy',
    ascending=False
)

print("Top1 accuracy:", results_df.head(1)['Test Accuracy'].iloc[0])
print("Top3 accuracy:", results_df.head(1)['Test Top3 Accuracy'].iloc[0])


# kNN Model

In [ ]:
X_train = pd.read_csv(data_path('X_train_scaled.csv'))
X_test = pd.read_csv(data_path('X_test_scaled.csv'))
Y_train = pd.read_csv(data_path('Y_train.csv')) 
Y_test = pd.read_csv(data_path('Y_test.csv'))

# since these are loaded as DataFrames convert into 1D arrays 
y_train = Y_train.squeeze()
y_test = Y_test.squeeze()

# we will also load in our label encoder
label_encoder = joblib.load(
    data_path('label_encoder.pkl')
)

# train baseline k=5 model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

genre_labels = label_encoder.classes_

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred, average='macro')

recall = recall_score(y_test, y_pred, average='macro')

f1 = f1_score(y_test, y_pred, average='macro')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=genre_labels,
    yticklabels=genre_labels
)

plt.title("kNN Confusion Matrix")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")

plt.savefig(figure_path('worse_knn_model.png'), dpi=300, bbox_inches="tight")

plt.show()

# try out diff k values 
k_values = range(1, 30)

train_accs = []
test_accs = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    train_pred = knn.predict(X_train)
    test_pred = knn.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

plt.figure(figsize=(8, 5))
plt.plot(k_values, train_accs, label='Train Accuracy')
plt.plot(k_values, test_accs, label='Test Accuracy')

plt.title("kNN Train/Test Accuracy vs k")
plt.xlabel("Number of Neighbors")
plt.ylabel("Accuracy Rate")
plt.legend()
plt.grid(True)

highest_acc = max(test_accs)

best_k = k_values[test_accs.index(highest_acc)]

print(f"Highest Test Accuracy: {highest_acc:.4f}")
print(f"Best k: {best_k}")

plt.savefig(figure_path('k-value-test-knn.png'), dpi=300, bbox_inches="tight")

plt.show()


#### Testing Different Distance Metrics 
This metric determines how we measure similarity between songs. Different metrics changes the shape of closeness. The default is Euclidean Distance that is an ordinary straight line distance. It is sensitive to large feature differences and works best when they are standarized, which we have done in preprocessing. It makes circular neighborhoods. 

There is also Manhattan Distance that adds absolute differences so it's less sensitive to outliers and more robust when some features are wildly different. It makes diamond-shaped neighborhoods. 

Finally there is Minkowski Distance which is just a generalized distance metric that has a parameter p involved which can make the model more or less sensitive to large features differences. A high p increases this. 

In [ ]:
metrics = ['euclidean', 'manhattan']

for metric in metrics:
    knn = KNeighborsClassifier(
        n_neighbors=10,
        metric=metric
    )

    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print(metric, acc)

p_values = [1, 2, 3, 4]

for p in p_values: 
    knn = KNeighborsClassifier(
        n_neighbors=10,
        metric='minkowski',
        p=p
    )
    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print(f"p={p}: {acc}")

# finding best k with manhattan distance 
k_values = range(1, 30)

train_accs = []
test_accs = []

for k in k_values:
    knn = KNeighborsClassifier(
        n_neighbors=k, 
        metric='manhattan'
        )
    knn.fit(X_train, y_train)

    train_pred = knn.predict(X_train)
    test_pred = knn.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

plt.figure(figsize=(8, 5))
plt.plot(k_values, train_accs, label='Train Accuracy')
plt.plot(k_values, test_accs, label='Test Accuracy')


plt.title("kNN Train/Test Accuracy vs k")
plt.xlabel("Number of Neighbors")
plt.ylabel("Accuracy Rate")
plt.legend()
plt.grid(True)

highest_acc = max(test_accs)

best_k = k_values[test_accs.index(highest_acc)]

print(f"Highest Test Accuracy: {highest_acc:.4f}")
print(f"Best k: {best_k}")

plt.savefig(figure_path('best-knn-model-accuracy.png'), dpi=300, bbox_inches="tight")

plt.show()

# retrain best model
best_knn = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='manhattan'
)

best_knn.fit(X_train, y_train)

# predictions from best model
best_pred = best_knn.predict(X_test)

# metrics
# metrics
accuracy = accuracy_score(y_test, best_pred)
precision = precision_score(y_test, best_pred, average='macro')
recall = recall_score(y_test, best_pred, average='macro')
f1 = f1_score(y_test, best_pred, average='macro')

print(f"Best k: {best_k}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# confusion matrix
cm = confusion_matrix(y_test, best_pred)

# genre labels
genre_labels = label_encoder.classes_

# plot confusion matrix
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=genre_labels,
    yticklabels=genre_labels
)

plt.title(f"kNN Confusion Matrix (k={best_k})")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")

plt.savefig(figure_path('best_knn_model.png'), dpi=300, bbox_inches="tight")

plt.show()


From this output it looks like using the Manhattan Distance overall slightly outperforms the regular Euclidean Distance. This could stem from the fact that some songs could differ mainly in just one feature, where Euclidean squares that difference and it starts to dominate. Manhatten just takes the absolute value of the difference and adds it so all features contribute more evenly. This can help since genres tend to have feature overlaps and no single feature should dominate similarity calculations. 

#### Final Results: 

Based on just the highest test accuracy k = 28 performs the best with the Manhattan Distance metric. The accuracy is about 52%. This signals to us that kNN is not the best model for this job since separating genre's based on features may be more complex than what kNN can achieve.

## PCA Projection

In [ ]:
# reduce 11 features into 2 principal components
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)

# make dataframe for plotting
pca_df = pd.DataFrame(
    X_train_pca,
    columns=["PC1", "PC2"]
)

# convert encoded labels back to genre names
pca_df["Genre"] = label_encoder.inverse_transform(y_train)

plt.figure(figsize=(10, 8))

for genre in pca_df["Genre"].unique():
    genre_data = pca_df[pca_df["Genre"] == genre]

    plt.scatter(
        genre_data["PC1"],
        genre_data["PC2"],
        label=genre,
        alpha=0.6,
        s=25
    )

plt.title("PCA Projection of Music Genres")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.2f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.2f}% variance)")
plt.legend(title="Genre", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)

plt.savefig(figure_path('PCA_projection.png'), dpi=300, bbox_inches="tight")
plt.show()


As we can see from this plot, condensing the 11 input features into 2 principal components only keeps abot 33% of the structure of the original 11D data. We can also see that the genres overlap a ton, making this a non-trivial problem. Rap and hip-hop appears to be completely overlapped and close to one another, while rock is further away from other genres and thus easier to classify, same with EDM.

In [ ]:
k_values = range(1, 30)

train_top3_accs = []
test_top3_accs = []

for k in k_values:

    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric='manhattan'
    )

    knn.fit(X_train, y_train)

    # predicted probabilities
    train_prob = knn.predict_proba(X_train)
    test_prob = knn.predict_proba(X_test)

    # top-3 accuracies
    train_top3_acc = top_k_accuracy_score(
        y_train,
        train_prob,
        k=3
    )

    test_top3_acc = top_k_accuracy_score(
        y_test,
        test_prob,
        k=3
    )


    train_top3_accs.append(train_top3_acc)
    test_top3_accs.append(test_top3_acc)

plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    train_top3_accs,
    label='Train Top-3 Accuracy'
)

plt.plot(
    k_values,
    test_top3_accs,
    label='Test Top-3 Accuracy'
)

plt.title("kNN Top-3 Accuracy vs k")
plt.xlabel("Number of Neighbors")
plt.ylabel("Top-3 Accuracy Rate")

plt.legend()
plt.grid(True)


highest_acc = max(test_top3_accs)
best_k = k_values[test_top3_accs.index(highest_acc)]

print(f"Highest Top-3 Test Accuracy: {highest_acc:.4f}")
print(f"Best k: {best_k}")

plt.savefig(figure_path('top-3-accuracy-knn.png'), dpi=300, bbox_inches="tight")

plt.show()


# SVM Model

In [ ]:
X_train = pd.read_csv(data_path('X_train_scaled.csv'))
X_test  = pd.read_csv(data_path('X_test_scaled.csv'))
Y_train = pd.read_csv(data_path('Y_train.csv')).values.ravel()
Y_test  = pd.read_csv(data_path('Y_test.csv')).values.ravel()

label_encoder = joblib.load(data_path('label_encoder.pkl'))

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Y_train:', Y_train.shape)
print('Y_test: ', Y_test.shape)
print('Classes:', label_encoder.classes_)

# baseline 
baseline_svm = SVC(kernel='rbf', random_state=0)
baseline_svm.fit(X_train, Y_train)

Y_pred_baseline = baseline_svm.predict(X_test)

baseline_train_acc = accuracy_score(Y_train, baseline_svm.predict(X_train))
baseline_test_acc  = accuracy_score(Y_test, Y_pred_baseline)

print(f'Baseline SVM (C=1.0, gamma=scale)')
print(f'Training accuracy: {baseline_train_acc:.4f}')
print(f'Test accuracy: {baseline_test_acc:.4f}')

# hyperparameter tuning
grid = [
    {'kernel': ['rbf'],    'C': [0.1, 1, 10], 'gamma': ['scale', 1, 0.1, 0.01]},
    {'kernel': ['poly'],   'C': [0.1, 1, 10], 'gamma': ['scale', 1, 0.1, 0.01], 'degree': [2, 3]},
    {'kernel': ['linear'], 'C': [0.1, 1, 10]},
]

grid_search = GridSearchCV(
    SVC(random_state=0),
    grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, Y_train)

print('Best parameters:', grid_search.best_params_)
print(f'Best cross validation accuracy: {grid_search.best_score_:.4f}')

results = pd.DataFrame(grid_search.cv_results_)

# build a readable label for each non-C combination
def label_row(row):
    parts = [row['param_kernel']]

    if pd.notna(row.get('param_gamma')):
        parts.append(f"gamma={row['param_gamma']}")

    if pd.notna(row.get('param_degree')):
        parts.append(f"deg={int(row['param_degree'])}")

    return ', '.join(parts)

results['combo'] = results.apply(label_row, axis=1)

# unique combos
combos = sorted(results['combo'].unique())

# generate enough visually distinct colors
cmap = plt.colormaps['tab20']
colors = [cmap(i / max(len(combos) - 1, 1)) for i in range(len(combos))]

# map each combo to a unique color
color_map = dict(zip(combos, colors))

plt.figure(figsize=(10, 6))

for combo in combos:
    sub = results[results['combo'] == combo].sort_values('param_C')

    plt.plot(
        sub['param_C'].astype(float),
        sub['mean_test_score'],
        marker='o',
        linewidth=2,
        color=color_map[combo],
        label=combo
    )

plt.xscale('log')

plt.title("Grid Search CV Accuracy vs C, by kernel/gamma/degree")
plt.xlabel("C")
plt.ylabel("Mean CV Accuracy")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    fontsize=8
)

plt.grid(True)
plt.tight_layout()
plt.show()

best_kernel = grid_search.best_params_['kernel']
best_C      = grid_search.best_params_['C']
best_gamma  = grid_search.best_params_['gamma']   # 'scale'
extra = {'degree': grid_search.best_params_['degree']} if best_kernel == 'poly' else {}

# sweep one parameter at a time, holding the other at the best model's value
C_sweep     = np.logspace(-2, 3, 13)   # 0.01 ... 1000
gamma_sweep = np.logspace(-4, 1, 15)   # 0.0001 ... 10

def sweep(param, values, fixed):
    train, test = [], []
    for v in values:
        model = SVC(kernel=best_kernel, random_state=0, **{param: v}, **fixed, **extra)
        model.fit(X_train, Y_train)
        train.append(accuracy_score(Y_train, model.predict(X_train)))
        test.append(accuracy_score(Y_test,  model.predict(X_test)))
    return train, test

def plot_sweep(ax, values, train, test, title, xlabel):
    ax.plot(values, train, marker='o', linewidth=2, label='Train')
    ax.plot(values, test,  marker='o', linewidth=2, label='Test')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

C_train, C_test = sweep('C', C_sweep, {'gamma': best_gamma})
plot_sweep(axes[0], C_sweep, C_train, C_test,
           f'Accuracy vs C  (gamma={best_gamma})', 'C (log scale)')

g_train, g_test = sweep('gamma', gamma_sweep, {'C': best_C})
plot_sweep(axes[1], gamma_sweep, g_train, g_test,
           f'Accuracy vs gamma  (C={best_C})', 'gamma (log scale)')

fig.suptitle(f'Train vs Test Accuracy Sweeps for {best_kernel.upper()} kernel', fontsize=14)
plt.tight_layout()
plt.show()

bi = int(np.argmax(C_test))
gi = int(np.argmax(g_test))
print(f'C sweep highest test accuracy: {C_test[bi]:.4f} at C = {C_sweep[bi]:.4g}')
print(f'gamma sweep highest test accuracy: {g_test[gi]:.4f} at gamma = {gamma_sweep[gi]:.4g}')

best_svm = grid_search.best_estimator_

Y_pred = best_svm.predict(X_test)

train_acc = accuracy_score(Y_train, best_svm.predict(X_train))
test_acc  = accuracy_score(Y_test, Y_pred)

print(f'Best SVM: {grid_search.best_params_}')
print(f'Training accuracy: {train_acc:.4f}')
print(f'Test accuracy: {test_acc:.4f}')
print()
print(classification_report(Y_test, Y_pred, target_names=label_encoder.classes_))

# confusion matrix
cm = confusion_matrix(Y_test, Y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title(f'Confusion Matrix for Best SVM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# sweep each RBF knob one at a time, holding the other at the best model's value
C_values     = np.logspace(-2, 2, 9)    # 0.01 ... 100
gamma_values = np.logspace(-4, 1, 11)   # 0.0001 ... 10

def top3_sweep(param, values, fixed):
    train, test = [], []
    for v in values:
        svm = SVC(kernel=best_kernel, probability=True, random_state=0,
                  **{param: v}, **fixed, **extra)
        svm.fit(X_train, Y_train)
        train.append(top_k_accuracy_score(Y_train, svm.predict_proba(X_train), k=3))
        test.append(top_k_accuracy_score(Y_test,  svm.predict_proba(X_test),  k=3))
    return train, test

def plot_sweep(ax, values, train, test, title, xlabel):
    ax.plot(values, train, marker='o', linewidth=2, label='Train Top-3 Accuracy')
    ax.plot(values, test,  marker='o', linewidth=2, label='Test Top-3 Accuracy')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Top-3 Accuracy Rate')
    ax.legend()
    ax.grid(True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

C_train, C_test = top3_sweep('C', C_values, {'gamma': best_gamma})
plot_sweep(axes[0], C_values, C_train, C_test,
           f'Top-3 Accuracy vs C  (gamma={best_gamma})', 'C (log scale)')

g_train, g_test = top3_sweep('gamma', gamma_values, {'C': best_C})
plot_sweep(axes[1], gamma_values, g_train, g_test,
           f'Top-3 Accuracy vs gamma  (C={best_C})', 'gamma (log scale)')

fig.suptitle(f'Top-3 Train vs Test Accuracy Sweeps for {best_kernel.upper()} kernel', fontsize=14)
plt.tight_layout()
plt.show()

bi = int(np.argmax(C_test))
gi = int(np.argmax(g_test))
print(f'C sweep highest Top-3 test accuracy: {C_test[bi]:.4f} at C = {C_values[bi]:.4g}')
print(f'gamma sweep highest Top-3 test accuracy: {g_test[gi]:.4f} at gamma = {gamma_values[gi]:.4g}')


We run a single grid search over all three kernels and their relevant parameters. GridSearchCV does 5-fold cross-validation on the training set for every combination and picks the one with the best mean CV accuracy.

The following are the hyperparameters that we test:
- **kernel:** defines the function used to transform data for separating classes; `linear` (straight), `rbf` (curved, general-purpose), `poly` (curved, flexibility set by `degree`).
- **C:** controls the trade-off between a wider margin (low C, risk underfitting) and correctly classifying all points (high C, risk overfitting).
- **gamma:** controls the influence of data points; high `gamma` leads to tight boundary that risks overfitting.
- **degree:** polynomial degree; high degree creates more complex curves but risks overfitting.

# MLP

This is the 4th model we are using to analyze our data. The goal is to compare each model's complexity and accuracy.

### Our approach:
*   Import the data with standardized file/path names to match our main document
*   Use the FFNN setup from HW3 as the baseline for this neural network
*   Implement random search using keras tuner, identifying the best hyperparameters
*   Create a second model using "top-k" accuracy
*   Plotting & comparing the models, to validate that the model is training as expected and that the results are fairly accurate
*   Quantify the model complexity and accuracy

### Improvements/optimizations attempted
*   Adding dropout layers to prevent model overfitting, especially at higher node counts
*   Testing out "top-k" accuracy, meaning the model is considered correct if one out of the top-k guesses correctly matches the music genre
*   Using random search to optimize the number of nodes in each layer, the learning rate, and the k in "top-k" analysis.


In [ ]:
X_train = pd.read_csv(data_path('X_train_scaled.csv'))
X_test = pd.read_csv(data_path('X_test_scaled.csv'))
Y_train = pd.read_csv(data_path('Y_train.csv'))
Y_test = pd.read_csv(data_path('Y_test.csv'))

# found the genre mapping from the preprocessing .ipynb file
genre_mapping = {
    0: 'edm',
    1: 'hiphop',
    2: 'latin',
    3: 'pop',
    4: 'r&b',
    5: 'rap',
    6: 'rock'
}

Y_train['Genre_Name'] = Y_train['Genre'].map(genre_mapping)
Y_test['Genre_Name'] = Y_test['Genre'].map(genre_mapping)

print("Unique Genres:", Y_train['Genre_Name'].unique())

def prior_model(hp):
  input_dim = 11
  # in order to allow keras_tuner to run, I have to leave the parameters defined as a testable range
  # coding the variable hp based on: https://medium.com/swlh/hyperparameter-tuning-in-keras-tensorflow-2-with-keras-tuner-randomsearch-hyperband-3e212647778f
  hidden_dim_1 = hp.Int('units_1', min_value=10, max_value=100, step=10)
  hidden_dim_2 = hp.Int('units_2', min_value=10, max_value=100, step=10)
  hidden_dim_3 = hp.Int('units_3', min_value=10, max_value=100, step=10)
  output_dim = 7
  learn_rate = hp.Float('learning_rate', min_value=0.001, max_value=0.5)

  old_model = Sequential()

  # when i use ReLU instead of sigmoid, i get actual classification
  # using sigmoid only makes this model fail, especially in confusion matrix later
  old_model.add(Dense(hidden_dim_1, input_dim=input_dim, activation='relu'))
  old_model.add(Dense(hidden_dim_2, activation='relu'))
  # droppout added based on following guide: https://machinelearningmastery.com/dropout-regularization-deep-learning-models-keras/
  old_model.add(Dropout(0.2))
  old_model.add(Dense(hidden_dim_3, activation='relu'))
  old_model.add(Dropout(0.2))
  old_model.add(Dense(output_dim, activation='softmax')) # as per Jake's Piazza instruction

  old_model.compile(loss='sparse_categorical_crossentropy', optimizer=SGD(learning_rate=learn_rate))
  return old_model

def find_hp(which_model, name):
  tuner = keras_tuner.RandomSearch(
      which_model,
      objective='val_loss',
      max_trials=10,
      overwrite=True,
      directory=str(DATA_DIR / 'keras_tuner'),
      project_name=name.replace(' ', '_').lower())
  # for the best-k searching as well
  tuner.search(X_train, Y_train['Genre'], epochs=10, validation_data=(X_test, Y_test['Genre']))
  best_model = tuner.get_best_models()[0]

  # getting the hyperparameters:
  best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

  best_dim_1 = best_hps.get('units_1')
  best_dim_2 = best_hps.get('units_2')
  best_dim_3 = best_hps.get('units_3')
  best_learn_rate = best_hps.get('learning_rate')

  print(f'{name} ***************')
  print(f'hidden_dim_1 = {best_dim_1}')
  print(f'hidden_dim_2 = {best_dim_2}')
  print(f'hidden_dim_3 = {best_dim_3}')
  print(f'learning_rate = {best_learn_rate}')
  if name == 'OPTIMIZATION WITH TOP-K ERROR':
    print(f'chosen K = {best_hps.get("TopK")}')
  print('*********************')

  return best_dim_1, best_dim_2, best_dim_3, best_learn_rate, best_model

best_dim_1, best_dim_2, best_dim_3, best_learn_rate, best_prior_model = find_hp(prior_model, 'NORMAL MODEL OPTIMIZATION')

" New model using top-K genre accuracy "

input_dim = 11
hidden_dim_1 = best_dim_1
hidden_dim_2 = best_dim_2
hidden_dim_3 = best_dim_3
output_dim = 7
learn_rate = best_learn_rate
k_defined = 3

model = Sequential()

# when i use ReLU instead of sigmoid, i get actual classification
# using sigmoid only makes this model fail, especially in confusion matrix later
model.add(Dense(hidden_dim_1, input_dim=input_dim, activation='relu'))
model.add(Dense(hidden_dim_2, activation='relu'))
# droppout added based on following guide: https://machinelearningmastery.com/dropout-regularization-deep-learning-models-keras/
model.add(Dropout(0.2))
model.add(Dense(hidden_dim_3, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(output_dim, activation='softmax')) # as per Jake's Piazza instruction

# implementing top-3 genre matching based on:
# https://medium.com/@juanfraleonhardt/music-genre-classification-a-machine-learning-exercise-9c83108fd2bb
sparse_topK_categorical_accuracy = keras.metrics.SparseTopKCategoricalAccuracy(k = k_defined)

model.compile(loss='sparse_categorical_crossentropy', optimizer=SGD(learning_rate=learn_rate), metrics=[sparse_topK_categorical_accuracy])


In [ ]:
# its own cell because it takes so long to run
epoch_number = 50

old_fitted_model = best_prior_model.fit(X_train, Y_train['Genre'], epochs=epoch_number, validation_data=(X_test, Y_test['Genre'])) #batch_size=128,


In [ ]:
# its own cell because it takes so long to run
best_fitted_model = model.fit(X_train, Y_train['Genre'], epochs=epoch_number, validation_data=(X_test, Y_test['Genre'])) #batch_size=128,


In [ ]:
fig,(ax1,ax2) = plt.subplots(1,2, figsize=(10,5))#, sharex=True, sharey=True)

epoch = np.arange(1, epoch_number+1)

old_train_mse = old_fitted_model.history['loss']
old_val_mse = old_fitted_model.history['val_loss']
ax1.plot(epoch, old_train_mse, color='blue', linestyle='-', label='Training MSE')
ax1.plot(epoch, old_val_mse, color='orange', linestyle='-', label='Testing MSE')
ax1.set_xlabel('epoch')
ax1.set_ylabel('MSE')
ax1.set_ylim(0.75, 2)
ax1.set_xlim(0, epoch_number+1)
ax1.set_title('Base model')

best_train_sparse_top_k = best_fitted_model.history['loss']
best_val_mse = best_fitted_model.history['val_loss']
topk_key = next((key for key in best_fitted_model.history.keys() if key.startswith('val_') and 'top' in key.lower()), None)
best_val_sparse_top_k = best_fitted_model.history[topk_key] if topk_key else None

ax2.plot(epoch, best_train_sparse_top_k, color='blue', linestyle='-', label='Training error')
ax2.plot(epoch, best_val_mse, color='orange', linestyle='-', label='Testing error')
# ax2.plot(epoch, best_val_sparse_top_k, color='green', linestyle='-', label='Testing SCC error')
ax2.set_xlabel('epoch')
ax2.set_ylabel('MSE')
ax2.set_ylim(0.75, 2)
ax2.set_xlim(0, epoch_number+1)
ax2.set_title('Base plus top-k model')


# plt.xlabel('Epoch')
# plt.ylabel('MSE')
plt.legend()
plt.show()

genre_labels = ['edm', 'hiphop', 'latin', 'pop', 'r&b', 'rap', 'rock']

def print_cm(model_name, name):

  Y_pred = model_name.predict(X_test)
  predictions = np.argmax(Y_pred, axis=1)

  accuracy = accuracy_score(Y_test['Genre'], predictions)
  precision = precision_score(Y_test['Genre'], predictions, average='macro')
  recall = recall_score(Y_test['Genre'], predictions, average='macro')
  f1 = f1_score(Y_test['Genre'], predictions, average='macro')

  print("Accuracy:", accuracy)
  print("Precision:", precision)
  print("Recall:", recall)
  print("F1:", f1)

  old_cm = confusion_matrix(Y_test['Genre'], predictions)

  plt.figure(figsize=(5, 5))

  sns.heatmap(
      old_cm,
      annot=True,
      fmt='d',
      cmap='Blues',
      xticklabels=genre_labels,
      yticklabels=genre_labels
  )

  plt.title(f"Confusion Matrix {name}")
  plt.xlabel("Predicted Genre")
  plt.ylabel("True Genre")

  plt.show()

print_cm(best_prior_model, 'old model')
print_cm(model, 'best model')

if topk_key:
  top3_acc_history = best_fitted_model.history[topk_key]
  best_top3_acc = max(top3_acc_history)
  print(f"best top-3 accuracy: {best_top3_acc:.4f}")
else:
  print("Top-3 validation accuracy was not found in the training history.")
